[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_51_Phase5_OSS_Architecture.ipynb)

# Lesson 51 — Phase 5 · Open-Source AI Project: Architecture & Scaffold

## What's Phase 5?

You've completed **50 lessons** across 5 tracks:

| Phase | Track | Lessons | Status |
|---|---|---|---|
| 4 — Track 1 | Reliability & Safety | L24–L31 | ✅ Complete |
| 4 — Track 2 | Multi-Agent Coordination | L32–L36 | ✅ Complete |
| 4 — Track 3 | Self-Hosted / Fine-Tuning | L37–L41 | ✅ Complete |
| 4 — Track 4 | Voice + Multimodal Agents | L42–L45 | ✅ Complete |
| 4 — Track 5 | Agent-Ops & Infra | L46–L50 | ✅ Complete |

**Phase 5 goal:** Take everything you've built and ship it as a real, production-quality open-source Python package — `auto_researcher` v2.0.

> "Open source to prove your worth" — this is the deliverable. A GitHub repo that demonstrates mastery of AI engineering at every layer: agents, reliability, observability, multimodal, fine-tuning, and deployment.

## What you'll build in this lesson

1. **Architecture design** — the complete v2.0 system integrating all 5 tracks
2. **Project scaffold** — the exact directory structure, generated in-notebook
3. **Config system** — a single `AutoResearcherConfig` Pydantic model that wires everything
4. **Component registry** — a dependency-injection pattern that makes testing clean
5. **Core pipeline** — a working `AutoResearcherV2` class running in Colab
6. **CLI + FastAPI** — the user-facing surface
7. **OSS packaging** — `pyproject.toml`, README template, GitHub Actions skeleton

## Phase 5 Roadmap

| Lesson | Topic | What you'll ship |
|---|---|---|
| **L51** ← you are here | Architecture & Scaffold | Project structure, config, registry, core pipeline |
| **L52** | Advanced Retrieval | Semantic caching + hybrid search at scale (sqlite-vec + HF embeddings) |
| **L53** | Production Deployment | Full `docker-compose` — app + vLLM + Jaeger + Prometheus + Grafana |
| **L54** | Developer Experience | Docs (mkdocs), contributing guide, pytest fixtures, coverage gate |
| **L55** | Capstone — Ship It | Integration smoke test, `gh release`, `twine upload`, README badges |

Each lesson adds a real, mergeable layer to the same repo. By L55, you have a publishable v2.0 release.

---

## The design constraint

> **Every component must be independently testable and optionally disableable.**

This one rule determines the entire architecture. It means:
- Reliability spine can run without the A2A swarm
- Observability can run in `noop` mode in unit tests
- Multimodal features degrade gracefully when API keys are absent
- vLLM backend is optional (falls back to Anthropic API)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
!pip install anthropic pydantic pydantic-settings typer fastapi uvicorn httpx              nest_asyncio rich tabulate opentelemetry-api opentelemetry-sdk -q

import os, sys, json, time, asyncio, textwrap, uuid
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Callable, Any

# Load Anthropic API key from Colab Secrets (set once, reused every lesson)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Running locally — set ANTHROPIC_API_KEY in your environment
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set ANTHROPIC_API_KEY in Colab Secrets or your environment")
    print("✅ API key found in environment")

import anthropic
import nest_asyncio
nest_asyncio.apply()

HAIKU   = "claude-haiku-4-5"
SONNET  = "claude-sonnet-4-6"
client  = anthropic.Anthropic()
print(f"anthropic SDK {anthropic.__version__} ready")

## The `auto_researcher` v2.0 Architecture

```
                        ┌─────────────────────────────────────────────────────┐
                        │              auto_researcher v2.0                   │
                        │                                                     │
   User / CLI           │   ┌─────────────────────────────────────────────┐   │
   ─────────►  FastAPI  │   │          AutoResearcherV2.research()        │   │
   /research  :8000     │   │                                             │   │
                        │   │  ┌──────────────┐   ┌──────────────────┐   │   │
                        │   │  │  Reliability  │   │   A2A Swarm      │   │   │
                        │   │  │  Spine (T1)   │   │   (Track 2)      │   │   │
                        │   │  │               │   │                  │   │   │
                        │   │  │ CircuitBreaker│   │ Searcher :9001   │   │   │
                        │   │  │ FallbackChain │◄──│ Critic   :9002   │   │   │
                        │   │  │ CostMeter     │   │ Synth    :9003   │   │   │
                        │   │  └──────┬────────┘   └──────────────────┘   │   │
                        │   │         │                                    │   │
                        │   │  ┌──────▼────────┐   ┌──────────────────┐   │   │
                        │   │  │  Inference    │   │  Observability   │   │   │
                        │   │  │  Backend      │   │  (Track 5)       │   │   │
                        │   │  │               │   │                  │   │   │
                        │   │  │ vLLM :8080    │   │ OTel Traces      │   │   │
                        │   │  │  ↓ fallback   │   │ Prometheus /mtrc │   │   │
                        │   │  │ Anthropic API │   │ Async Eval Queue │   │   │
                        │   │  └───────────────┘   └──────────────────┘   │   │
                        │   │                                             │   │
                        │   │  ┌──────────────┐   ┌──────────────────┐   │   │
                        │   │  │  Retrieval   │   │  Multimodal      │   │   │
                        │   │  │  (L52)       │   │  (Track 4)       │   │   │
                        │   │  │              │   │                  │   │   │
                        │   │  │ sqlite-vec   │   │ PDF / Image /    │   │   │
                        │   │  │ BM25 + RRF   │   │ Voice (optional) │   │   │
                        │   │  └──────────────┘   └──────────────────┘   │   │
                        │   └─────────────────────────────────────────────┘   │
                        └─────────────────────────────────────────────────────┘
```

### Key design decisions

| Decision | Choice | Why |
|---|---|---|
| Config | `pydantic-settings` | Env vars, `.env` files, type safety |
| Component wiring | Manual DI (no framework) | Readable, testable, no magic |
| Async | `asyncio` throughout | Matches A2A, OTel, eval queue |
| Observability | `noop` by default | Tests don't need Jaeger/Prometheus running |
| Inference | `InferenceBackend` ABC | Swap Anthropic ↔ vLLM without changing callers |
| Swarm | Optional (flag) | Single-process mode for low-cost tasks |

In [ ]:
# ── Project Scaffold ─────────────────────────────────────────────────────────
# Generates the full v2.0 directory tree in /content/auto_researcher_v2/
# This is the canonical layout for the OSS repo.

import os
from pathlib import Path

ROOT = Path("/content/auto_researcher_v2")

SCAFFOLD = {
    # Core package
    "auto_researcher/__init__.py":           """""""""" ,
    "auto_researcher/config.py":              """"""# Config lives here (built in this lesson)""",
    "auto_researcher/pipeline.py":            """"""# Core research pipeline (built in this lesson)""",
    "auto_researcher/registry.py":            """"""# Component registry (built in this lesson)""",

    # Reliability (from Track 1 / L24-L31)
    "auto_researcher/reliability/__init__.py": """""""""",
    "auto_researcher/reliability/breakers.py": """"""# CircuitBreaker — ported from L30""",
    "auto_researcher/reliability/fallback.py": """"""# FallbackChain — ported from L30""",
    "auto_researcher/reliability/cost.py":     """"""# CostMeter — ported from L22""",

    # Inference backends (from Track 3 / L37)
    "auto_researcher/inference/__init__.py":   """""""""",
    "auto_researcher/inference/base.py":       """"""# InferenceBackend ABC""",
    "auto_researcher/inference/anthropic.py":  """"""# AnthropicBackend — default""",
    "auto_researcher/inference/vllm.py":       """"""# VLLMBackend — optional self-hosted""",

    # Retrieval (built in L52)
    "auto_researcher/retrieval/__init__.py":   """""""""",
    "auto_researcher/retrieval/store.py":      """"""# VectorStore ABC + SqliteVecStore""",
    "auto_researcher/retrieval/cache.py":      """"""# SemanticCache — new in L52""",

    # Swarm (from Track 2 / L32-L36)
    "auto_researcher/swarm/__init__.py":       """""""""",
    "auto_researcher/swarm/agents.py":         """"""# Searcher / Critic / Synthesizer A2A agents""",
    "auto_researcher/swarm/orchestrator.py":   """"""# Blackboard + Controller""",

    # Observability (from Track 5 / L46-L50)
    "auto_researcher/observability/__init__.py": """""""""",
    "auto_researcher/observability/tracing.py": """"""# OTel tracer setup""",
    "auto_researcher/observability/metrics.py": """"""# Prometheus metrics""",
    "auto_researcher/observability/eval.py":    """"""# AsyncEvalPipeline""",

    # Multimodal (from Track 4 / L42-L45)
    "auto_researcher/multimodal/__init__.py":  """""""""",
    "auto_researcher/multimodal/documents.py": """"""# DocumentLayer — PDF handling""",
    "auto_researcher/multimodal/vision.py":    """"""# Vision analysis""",

    # Surfaces
    "auto_researcher/api.py":                  """"""# FastAPI app""",
    "auto_researcher/cli.py":                  """"""# Typer CLI""",

    # Tests
    "tests/__init__.py":                       """""""""",
    "tests/test_config.py":                    """"""# Config validation tests""",
    "tests/test_pipeline.py":                  """"""# Pipeline integration tests""",
    "tests/test_reliability.py":               """"""# Reliability component tests""",
    "tests/conftest.py":                       """"""# Shared pytest fixtures""",

    # Evals
    "evals/golden_regression.jsonl":           """""""""",
    "evals/run_evals.py":                      """"""# CI eval runner""",

    # Docs (built in L54)
    "docs/index.md":                           """"""# Welcome to auto_researcher""",
    "docs/architecture.md":                    """"""# Architecture guide""",
    "docs/quickstart.md":                      """"""# 5-minute quickstart""",

    # Config / packaging
    ".env.example":                            """"""ANTHROPIC_API_KEY=sk-ant-...
VLLM_BASE_URL=
SWARM_ENABLED=false
OBSERVABILITY_ENABLED=false""",
    "pyproject.toml":                          """""""""",
    "README.md":                               """""""""",
    ".github/workflows/ci.yml":                """""""""",
    "docker-compose.yml":                      """""""""",
}

# Create the scaffold
for rel_path, content in SCAFFOLD.items():
    full = ROOT / rel_path
    full.parent.mkdir(parents=True, exist_ok=True)
    if not full.exists():
        full.write_text(content)

# Print the tree
print("auto_researcher_v2/")
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        rel = p.relative_to(ROOT)
        depth = len(rel.parts) - 1
        print("  " * depth + "├── " + p.name)

print(f"\n✅ Scaffold created at {ROOT}")
print(f"   {sum(1 for p in ROOT.rglob('*') if p.is_file())} files, {sum(1 for p in ROOT.rglob('*') if p.is_dir())} directories")

## The Config System

The single most important file in v2.0 is `config.py`. Every component reads from it — no magic strings scattered through the codebase.

### Why `pydantic-settings`?

```python
# Without it: scattered, brittle, untestable
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")  # buried in every file
VLLM_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8080")

# With it: one source of truth, type-safe, testable
config = AutoResearcherConfig()
backend = AnthropicBackend(config.anthropic_api_key)
```

The config also encodes **feature flags** — the mechanism that makes every component independently disableable.

```
SWARM_ENABLED=true          → spins up A2A swarm
OBSERVABILITY_ENABLED=true  → attaches OTel traces + Prometheus
VLLM_BASE_URL=...           → uses vLLM instead of Anthropic for worker calls
MULTIMODAL_ENABLED=true     → enables PDF / image processing
SEMANTIC_CACHE_ENABLED=true → enables vector-similarity cache (L52)
```

All flags default to `false` or empty so a fresh `pip install auto-researcher` "just works" with only an Anthropic key.

In [ ]:
# ── Config System ────────────────────────────────────────────────────────────
# Production version uses pydantic-settings; for Colab we use plain Pydantic.

from pydantic import BaseModel, Field, field_validator
from typing import Optional, Literal
import os

class ReliabilityConfig(BaseModel):
    fail_threshold: int    = 4
    cooldown_s: float      = 30.0
    max_cost_usd: float    = 1.0            # per-request budget cap
    fallback_models: list  = Field(default_factory=lambda: [HAIKU, "static"])

class SwarmConfig(BaseModel):
    enabled: bool          = False
    searcher_port: int     = 9001
    critic_port: int       = 9002
    synth_port: int        = 9003
    max_rounds: int        = 3

class ObservabilityConfig(BaseModel):
    enabled: bool          = False
    otlp_endpoint: str     = "http://localhost:4317"   # Jaeger / Grafana Tempo
    prometheus_port: int   = 8001
    eval_sample_rate: float = 0.3
    eval_model: str        = HAIKU

class InferenceConfig(BaseModel):
    backend: Literal["anthropic", "vllm"] = "anthropic"
    vllm_base_url: str     = "http://localhost:8080/v1"
    orchestrator_model: str = SONNET          # for planning & synthesis
    worker_model: str       = HAIKU           # for search, critique, expansion

class RetrievalConfig(BaseModel):
    enabled: bool          = True
    db_path: str           = ":memory:"      # sqlite-vec path; :memory: = ephemeral
    top_k: int             = 5
    hybrid_alpha: float    = 0.7              # weight for dense vs BM25 in RRF
    semantic_cache_enabled: bool = False      # new in L52
    cache_similarity_threshold: float = 0.92

class AutoResearcherConfig(BaseModel):
    """Single source of truth for all configuration."""
    anthropic_api_key: str      = Field(default_factory=lambda: os.environ["ANTHROPIC_API_KEY"])
    log_level: str              = "INFO"
    max_steps: int              = 10

    reliability: ReliabilityConfig    = Field(default_factory=ReliabilityConfig)
    swarm: SwarmConfig                = Field(default_factory=SwarmConfig)
    observability: ObservabilityConfig = Field(default_factory=ObservabilityConfig)
    inference: InferenceConfig        = Field(default_factory=InferenceConfig)
    retrieval: RetrievalConfig        = Field(default_factory=RetrievalConfig)

    @classmethod
    def from_env(cls) -> "AutoResearcherConfig":
        """Load from environment variables (production path)."""
        return cls(
            inference=InferenceConfig(
                backend="vllm" if os.getenv("VLLM_BASE_URL") else "anthropic",
                vllm_base_url=os.getenv("VLLM_BASE_URL", "http://localhost:8080/v1"),
            ),
            swarm=SwarmConfig(enabled=os.getenv("SWARM_ENABLED","false").lower()=="true"),
            observability=ObservabilityConfig(
                enabled=os.getenv("OBSERVABILITY_ENABLED","false").lower()=="true",
            ),
        )

# Verify it works
cfg = AutoResearcherConfig()
print("AutoResearcherConfig:")
print(f"  inference backend : {cfg.inference.backend}")
print(f"  orchestrator model: {cfg.inference.orchestrator_model}")
print(f"  worker model      : {cfg.inference.worker_model}")
print(f"  swarm enabled     : {cfg.swarm.enabled}")
print(f"  observability     : {cfg.observability.enabled}")
print(f"  retrieval db      : {cfg.retrieval.db_path}")
print(f"  max cost/req      : ${cfg.reliability.max_cost_usd:.2f}")

# 💡 EXPERIMENT: Try AutoResearcherConfig.from_env() after setting
#   os.environ["SWARM_ENABLED"] = "true"
#   and see how the config changes.

In [ ]:
# ── Component Registry ───────────────────────────────────────────────────────
# The registry is a lightweight DI container.
# It builds components once and caches them — each component reads from config.
# In tests, you swap individual components without touching others.

from dataclasses import dataclass, field
from functools import cached_property
from typing import Protocol, runtime_checkable

# ── InferenceBackend ABC ────────────────────────────────────────────────────
@runtime_checkable
class InferenceBackend(Protocol):
    """Any object that satisfies this protocol can be used as an inference backend."""
    def complete(self, system: str, user: str, *, model: str = "", max_tokens: int = 1024) -> str:
        ...
    def cost_usd(self) -> float:
        ...

class AnthropicBackend:
    """Direct Anthropic API — the default backend."""    def __init__(self, api_key: str):
        self._client = anthropic.Anthropic(api_key=api_key)
        self._total_cost = 0.0

    def complete(self, system: str, user: str, *, model: str = HAIKU, max_tokens: int = 1024) -> str:
        PRICES = {HAIKU: (0.80, 4.00), SONNET: (3.00, 15.00)}
        r = self._client.messages.create(
            model=model, max_tokens=max_tokens,
            system=system, messages=[{"role": "user", "content": user}]
        )
        inp, out = PRICES.get(model, (1.0, 5.0))
        self._total_cost += (r.usage.input_tokens * inp + r.usage.output_tokens * out) / 1_000_000
        return r.content[0].text

    def cost_usd(self) -> float:
        return self._total_cost

# ── ObservabilityBus (noop by default) ──────────────────────────────────────
class ObservabilityBus:
    """Records spans, metrics, and eval samples. Noop when disabled."""    def __init__(self, enabled: bool = False):
        self.enabled = enabled
        self._spans: list[dict] = []
        self._metrics: dict[str, float] = {}

    def span(self, name: str, **attrs):
        """Context manager that records a span."""        class _Span:
            def __init__(s): s.start = time.time()
            def __enter__(s): return s
            def __exit__(s, *_):
                if self.enabled:
                    self._spans.append({"name": name, "latency_ms": (time.time()-s.start)*1000, **attrs})
        return _Span()

    def record(self, key: str, value: float):
        if self.enabled:
            self._metrics[key] = self._metrics.get(key, 0) + value

    def summary(self) -> dict:
        return {"spans": len(self._spans), "metrics": self._metrics}

# ── CostMeter ───────────────────────────────────────────────────────────────
@dataclass
class CostMeter:
    _costs: dict = field(default_factory=dict)

    def record(self, tag: str, usd: float):
        self._costs[tag] = self._costs.get(tag, 0.0) + usd

    def total(self) -> float:
        return sum(self._costs.values())

    def report(self) -> dict:
        return dict(sorted(self._costs.items(), key=lambda x: -x[1]))

# ── ComponentRegistry ───────────────────────────────────────────────────────
class ComponentRegistry:
    """Builds and caches all components from config.
    In tests: override individual attributes before calling pipeline.
    """    def __init__(self, config: AutoResearcherConfig):
        self.config = config
        self._inference: Optional[InferenceBackend] = None
        self._obs: Optional[ObservabilityBus] = None
        self._meter: Optional[CostMeter] = None

    @property
    def inference(self) -> InferenceBackend:
        if self._inference is None:
            if self.config.inference.backend == "vllm":
                raise NotImplementedError("VLLMBackend: see auto_researcher/inference/vllm.py (built in L53)")
            self._inference = AnthropicBackend(self.config.anthropic_api_key)
        return self._inference

    @property
    def obs(self) -> ObservabilityBus:
        if self._obs is None:
            self._obs = ObservabilityBus(enabled=self.config.observability.enabled)
        return self._obs

    @property
    def meter(self) -> CostMeter:
        if self._meter is None:
            self._meter = CostMeter()
        return self._meter

# Test the registry
registry = ComponentRegistry(cfg)
print("ComponentRegistry initialized:")
print(f"  inference: {type(registry.inference).__name__}")
print(f"  obs noop : {not registry.obs.enabled}")
print(f"  meter    : CostMeter ready")

# 💡 EXPERIMENT: Swap the inference backend in tests:
#   class MockBackend:
#       def complete(self, *a, **kw): return "mocked answer"
#       def cost_usd(self): return 0.0
#   registry._inference = MockBackend()

## How a query flows through the system

```
User query: "Explain transformer attention mechanisms"

                AutoResearcherV2.research(query)
                          │
              ┌───────────▼──────────────┐
              │     1. Plan (Sonnet)      │ → 3 sub-questions
              └───────────┬──────────────┘
                          │
              ┌───────────▼──────────────┐
              │  2. Search (parallel)     │ → run_search(q1), run_search(q2), run_search(q3)
              │     asyncio.gather()      │   [each is a Haiku call or A2A Searcher agent]
              └───────────┬──────────────┘
                          │
              ┌───────────▼──────────────┐
              │  3. Draft (Sonnet)        │ → synthesize search results into draft
              └───────────┬──────────────┘
                          │
              ┌───────────▼──────────────┐
              │  4. Critique (Haiku)      │ → score quality, identify gaps
              └───────────┬──────────────┘
                          │  (if score < threshold → revise up to N times)
              ┌───────────▼──────────────┐
              │  5. Final answer          │ → return ResearchResult
              └───────────┬──────────────┘
                          │
          ┌───────────────┴───────────────────┐
          ▼                                   ▼
  OTel span recorded                  AsyncEvalPipeline.enqueue()
  Prometheus counter +1               (async judge, score stored)
```

Each step is wrapped in the reliability spine: **circuit breaker → fallback chain → cost meter**.

The swarm (when `enabled=True`) replaces steps 2–4 with actual A2A agents on separate ports. When `enabled=False`, the same logic runs in-process. **The caller can't tell the difference** — same `ResearchResult` shape either way.

In [ ]:
# ── AutoResearcherV2 Core Pipeline ───────────────────────────────────────────

from pydantic import BaseModel
from typing import List, Optional
import asyncio

class ResearchResult(BaseModel):
    query: str
    plan: List[str]           # sub-questions
    sources: List[str]        # raw search results
    draft: str                # synthesized answer
    critique_score: float     # 0-1
    final_answer: str
    cost_usd: float
    latency_s: float
    model_calls: int

class AutoResearcherV2:
    """
    The unified pipeline. All optional capabilities are gated by config flags.
    Swarm=False → fast, low-cost, single-process.
    Swarm=True  → full multi-agent A2A swarm (requires agents to be running).
    """
    def __init__(self, config: Optional[AutoResearcherConfig] = None,
                 registry: Optional[ComponentRegistry] = None):
        self.config = config or AutoResearcherConfig()
        self.registry = registry or ComponentRegistry(self.config)
        self._call_count = 0

    # ── Private helpers ─────────────────────────────────────────────────────

    def _infer(self, system: str, user: str, model: str = "") -> str:
        """Single chokepoint for all LLM calls — meters cost, records span."""        if not model:
            model = self.config.inference.worker_model
        self._call_count += 1
        with self.registry.obs.span("llm.call", model=model):
            text = self.registry.inference.complete(system, user, model=model)
        self.registry.meter.record(model, self.registry.inference.cost_usd())
        return text

    def _plan(self, query: str) -> List[str]:
        """Break query into 3 focused sub-questions."""        system = "You are a research planner. Given a query, return exactly 3 numbered sub-questions that together cover it fully. Be concise."
        raw = self._infer(system, f"Query: {query}", model=self.config.inference.orchestrator_model)
        lines = [l.strip().lstrip("123. ").strip() for l in raw.strip().split("\n") if l.strip()]
        return lines[:3] if len(lines) >= 3 else lines + [query] * (3 - len(lines))

    def _search(self, sub_question: str) -> str:
        """Simulated search: in production, calls Searcher A2A agent or a real search tool."""        system = "You are a research assistant. Answer the given sub-question factually and concisely in 2-3 sentences. Cite no specific URLs — just provide the key facts."
        return self._infer(system, sub_question)

    def _draft(self, query: str, sources: List[str]) -> str:
        """Synthesize search results into a coherent answer."""        src_block = "\n\n".join(f"[{i+1}] {s}" for i, s in enumerate(sources))
        system = "You are a research synthesizer. Write a clear, structured answer to the query based on the provided sources. Use markdown headers. Reference sources as [1], [2], [3]."
        return self._infer(
            system,
            f"Query: {query}\n\nSources:\n{src_block}",
            model=self.config.inference.orchestrator_model
        )

    def _critique(self, draft: str) -> float:
        """Score draft quality 0-10, return normalized 0-1 score."""        system = """You are a quality judge. Score the research answer on:
- Accuracy (0-4): factual correctness
- Completeness (0-3): all key aspects covered
- Clarity (0-3): well-structured and readable
Reply with ONLY a single integer 0-10 on the first line."""        raw = self._infer(system, f"Answer to evaluate:\n{draft}")
        try:
            score = int(raw.strip().split()[0])
            return min(max(score, 0), 10) / 10.0
        except Exception:
            return 0.7  # default if parse fails

    # ── Public API ──────────────────────────────────────────────────────────

    async def research(self, query: str) -> ResearchResult:
        """Main pipeline: plan → parallel search → draft → critique → return."""        t0 = time.time()
        self._call_count = 0

        with self.registry.obs.span("pipeline.research", query=query[:50]):
            # Step 1: Plan
            with self.registry.obs.span("pipeline.plan"):
                plan = self._plan(query)

            # Step 2: Parallel search (asyncio.gather over sync calls)
            # In production with swarm=True, these become A2A calls to Searcher :9001
            with self.registry.obs.span("pipeline.search", n_subqueries=len(plan)):
                loop = asyncio.get_event_loop()
                sources = await asyncio.gather(*[
                    loop.run_in_executor(None, self._search, sq)
                    for sq in plan
                ])

            # Step 3: Draft
            with self.registry.obs.span("pipeline.draft"):
                draft = self._draft(query, list(sources))

            # Step 4: Critique (with simple revision loop)
            score = self._critique(draft)
            if score < 0.65:
                # One revision pass
                system = "Improve this research answer for accuracy and completeness. Keep markdown structure."
                draft = self._infer(system,
                    f"Original query: {query}\n\nAnswer to improve:\n{draft}",
                    model=self.config.inference.orchestrator_model)
                score = self._critique(draft)

        total_cost = self.registry.meter.total()
        self.registry.obs.record("requests_total", 1)
        self.registry.obs.record("cost_usd", total_cost)

        return ResearchResult(
            query=query,
            plan=plan,
            sources=list(sources),
            draft=draft,
            critique_score=round(score, 2),
            final_answer=draft,
            cost_usd=round(total_cost, 5),
            latency_s=round(time.time() - t0, 2),
            model_calls=self._call_count,
        )

print("AutoResearcherV2 defined ✅")
print("Key methods: research(query) → ResearchResult")

In [ ]:
# ── Live Demo ────────────────────────────────────────────────────────────────

ar = AutoResearcherV2(config=AutoResearcherConfig())

# Run a query
result = asyncio.run(ar.research("How does RAG (Retrieval-Augmented Generation) improve LLM accuracy?"))

print("=" * 60)
print(f"Query: {result.query}")
print()
print("📋 PLAN (sub-questions):")
for i, q in enumerate(result.plan, 1):
    print(f"  {i}. {q}")

print()
print("📝 FINAL ANSWER (excerpt):")
print(textwrap.fill(result.final_answer[:400] + "...", width=70))

print()
print("📊 STATS:")
print(f"  Quality score   : {result.critique_score:.0%}")
print(f"  LLM calls       : {result.model_calls}")
print(f"  Cost            : ${result.cost_usd:.5f}")
print(f"  Latency         : {result.latency_s:.1f}s")
print()
print("💰 Cost breakdown:")
for model, cost in ar.registry.meter.report().items():
    print(f"  {model:30s}: ${cost:.5f}")

# 💡 EXPERIMENT: Enable observability and see the spans:
#   cfg2 = AutoResearcherConfig(observability=ObservabilityConfig(enabled=True))
#   ar2 = AutoResearcherV2(config=cfg2)
#   asyncio.run(ar2.research("What is chain-of-thought prompting?"))
#   print(ar2.registry.obs.summary())

In [ ]:
# ── CLI Interface ────────────────────────────────────────────────────────────
# In production: `auto-researcher research "my query"`
# Here we define the CLI and show what it looks like when run.

CLI_CODE = '''#!/usr/bin/env python3
"""auto-researcher CLI — powered by auto_researcher v2.0"""

import typer, asyncio, json, os
from auto_researcher.config import AutoResearcherConfig
from auto_researcher.pipeline import AutoResearcherV2

app = typer.Typer(help="AI-powered research assistant")

@app.command()
def research(
    query: str = typer.Argument(..., help="Research question"),
    json_out: bool = typer.Option(False, "--json", help="Output as JSON"),
    swarm: bool = typer.Option(False, "--swarm", help="Enable A2A swarm"),
    max_cost: float = typer.Option(0.10, "--max-cost", help="Max cost USD"),
):
    """Research a topic and print the answer."""
    from auto_researcher.config import ReliabilityConfig, SwarmConfig
    config = AutoResearcherConfig(
        reliability=ReliabilityConfig(max_cost_usd=max_cost),
        swarm=SwarmConfig(enabled=swarm),
    )
    ar = AutoResearcherV2(config=config)
    result = asyncio.run(ar.research(query))

    if json_out:
        typer.echo(result.model_dump_json(indent=2))
    else:
        typer.echo(f"\\n{result.final_answer}")
        typer.echo(f"\\n---")
        typer.echo(f"Score: {result.critique_score:.0%}  Cost: ${result.cost_usd:.4f}  Time: {result.latency_s:.1f}s")

@app.command()
def serve(
    port: int = typer.Option(8000, "--port"),
    swarm: bool = typer.Option(False, "--swarm"),
):
    """Start the FastAPI server."""
    import uvicorn
    from auto_researcher.api import create_app
    config = AutoResearcherConfig(swarm=SwarmConfig(enabled=swarm))
    uvicorn.run(create_app(config), host="0.0.0.0", port=port)

if __name__ == "__main__":
    app()
'''

# Write the CLI to the scaffold
(ROOT / "auto_researcher" / "cli.py").write_text(CLI_CODE)

print("CLI written to auto_researcher/cli.py")
print()
print("Usage examples:")
print("  auto-researcher research \"What is RAG?\" ")
print("  auto-researcher research \"Compare LLM training approaches\" --json")
print("  auto-researcher research \"Multi-agent coordination\" --swarm --max-cost 0.50")
print("  auto-researcher serve --port 8080 --swarm")
print()
print("In pyproject.toml, this maps to:")
print('  [project.scripts]')
print('  auto-researcher = "auto_researcher.cli:app"')

In [ ]:
# ── FastAPI App + pyproject.toml ─────────────────────────────────────────────

FASTAPI_CODE = '''
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import asyncio, json
from auto_researcher.config import AutoResearcherConfig
from auto_researcher.pipeline import AutoResearcherV2

class ResearchRequest(BaseModel):
    query: str
    stream: bool = False

def create_app(config: AutoResearcherConfig | None = None) -> FastAPI:
    config = config or AutoResearcherConfig.from_env()
    ar = AutoResearcherV2(config=config)
    app = FastAPI(title="auto-researcher", version="2.0.0")

    @app.post("/research")
    async def research(req: ResearchRequest):
        try:
            result = await ar.research(req.query)
            return result.model_dump()
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    @app.get("/health")
    def health():
        return {"status": "ok", "version": "2.0.0"}

    @app.get("/metrics")
    def metrics():
        return ar.registry.meter.report()

    return app
'''

PYPROJECT = '''[build-system]
requires = ["setuptools>=70", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "auto-researcher"
version = "2.0.0"
description = "Production AI research agent — reliable, observable, multimodal"
readme = "README.md"
license = { text = "MIT" }
requires-python = ">=3.10"
authors = [{ name = "Gourav Khanijoe", email = "gouravkhanijoe@gmail.com" }]
keywords = ["ai", "agents", "llm", "research", "anthropic"]
classifiers = [
    "Development Status :: 4 - Beta",
    "Intended Audience :: Developers",
    "License :: OSI Approved :: MIT License",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
    "Topic :: Scientific/Engineering :: Artificial Intelligence",
]
dependencies = [
    "anthropic>=0.40",
    "pydantic>=2.0",
    "pydantic-settings>=2.0",
    "fastapi>=0.110",
    "uvicorn[standard]>=0.29",
    "httpx>=0.27",
    "typer>=0.12",
    "rich>=13",
    "opentelemetry-api>=1.25",
    "opentelemetry-sdk>=1.25",
]

[project.optional-dependencies]
multimodal = ["pdfplumber", "pillow", "openai", "openai-whisper"]
vllm       = ["openai"]   # uses OpenAI-compatible vLLM endpoint
all        = ["auto-researcher[multimodal,vllm]"]

[project.scripts]
auto-researcher = "auto_researcher.cli:app"

[project.urls]
Homepage      = "https://github.com/gouravkhanijoe/auto-researcher"
Documentation = "https://gouravkhanijoe.github.io/auto-researcher"
Repository    = "https://github.com/gouravkhanijoe/auto-researcher.git"
Issues        = "https://github.com/gouravkhanijoe/auto-researcher/issues"

[tool.pytest.ini_options]
asyncio_mode = "auto"
testpaths    = ["tests"]

[tool.ruff]
line-length = 100
target-version = "py310"
'''

(ROOT / "auto_researcher" / "api.py").write_text(FASTAPI_CODE)
(ROOT / "pyproject.toml").write_text(PYPROJECT)

print("✅ api.py and pyproject.toml written")
print()
print("Install commands:")
print("  pip install auto-researcher                    # core only")
print("  pip install auto-researcher[multimodal]        # + PDF / voice / image")
print("  pip install auto-researcher[vllm]              # + vLLM backend")
print("  pip install auto-researcher[all]               # everything")
print()
print("Optional deps design rule:")
print("  → core never imports pdfplumber, whisper, openai")
print("  → multimodal/ subpackage imports them only when MULTIMODAL_ENABLED=true")
print("  → missing optional dep → graceful degradation, not ImportError at startup")

In [ ]:
# ── GitHub Actions CI ────────────────────────────────────────────────────────

CI_YAML = '''name: CI

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]
  schedule:
    - cron: "0 6 * * *"   # nightly golden regression

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "${{ matrix.python-version }}" }
      - run: pip install -e ".[all]" pytest pytest-asyncio -q
      - run: pytest tests/ -v --tb=short
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}

  eval-gate:
    runs-on: ubuntu-latest
    needs: test
    if: github.event_name == \'schedule\'
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -e . -q
      - run: python evals/run_evals.py --gate
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}

  release:
    runs-on: ubuntu-latest
    needs: [test]
    if: startsWith(github.ref, \'refs/tags/v\')
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install build twine -q
      - run: python -m build
      - run: twine upload dist/*
        env:
          TWINE_USERNAME: __token__
          TWINE_PASSWORD: ${{ secrets.PYPI_API_TOKEN }}
'''

(ROOT / ".github" / "workflows").mkdir(parents=True, exist_ok=True)
(ROOT / ".github" / "workflows" / "ci.yml").write_text(CI_YAML)

# Minimal conftest.py
CONFTEST = '''import pytest
from auto_researcher.config import AutoResearcherConfig
from auto_researcher.registry import ComponentRegistry

class MockInferenceBackend:
    """Zero-cost backend for unit tests. No API calls."""\
    _response = "This is a mock LLM response for testing."\
    def complete(self, system, user, **kw): return self._response\
    def cost_usd(self): return 0.0\
\
@pytest.fixture\
def test_config():\
    return AutoResearcherConfig()\
\
@pytest.fixture\
def test_registry(test_config):\
    registry = ComponentRegistry(test_config)\
    registry._inference = MockInferenceBackend()\
    return registry\
'''

(ROOT / "tests" / "conftest.py").write_text(CONFTEST)

# Minimal test_pipeline.py
TEST_CODE = '''import pytest, asyncio
from auto_researcher.pipeline import AutoResearcherV2
from auto_researcher.config import AutoResearcherConfig

@pytest.mark.asyncio
async def test_research_returns_result(test_registry):
    ar = AutoResearcherV2(registry=test_registry)
    result = await ar.research("What is machine learning?")
    assert result.query == "What is machine learning?"
    assert len(result.plan) == 3
    assert result.cost_usd == 0.0   # mock backend, no cost
    assert result.final_answer      # non-empty

@pytest.mark.asyncio
async def test_research_low_cost():
    """Integration test — hits real API. Skip in unit test runs."""\
    import os\
    if not os.environ.get("ANTHROPIC_API_KEY"): pytest.skip("No API key")\
    ar = AutoResearcherV2()\
    result = await ar.research("What is a transformer?")\
    assert result.cost_usd < 0.05   # should never exceed 5 cents\
    assert result.critique_score > 0.5\
'''

(ROOT / "tests" / "test_pipeline.py").write_text(TEST_CODE)

print("✅ CI workflow, conftest.py, and test_pipeline.py written")
print()
print("CI pipeline:")
print("  push/PR   → matrix test (py 3.10, 3.11, 3.12)")
print("  nightly   → eval-gate (golden regression suite)")
print("  git tag   → auto-release to PyPI")
print()
print("Test command:")
print("  pytest tests/ -v                    # runs unit tests (mock backend, free)")
print("  pytest tests/ -v -k integration     # runs API tests (costs money)")

## 10 Integration Pitfalls

When wiring 5 tracks together, these are the traps that kill projects:

| # | Pitfall | What goes wrong | Fix |
|---|---|---|---|
| 1 | **Optional dep at startup** | `import pdfplumber` at module level crashes for users who didn't install `[multimodal]` | Lazy import inside the function that needs it; wrap in `try/except ImportError` |
| 2 | **Config sprawl** | Each component has its own env vars (`BREAKER_THRESHOLD`, `VLLM_URL`, `OTLP_ENDPOINT`...) | One `AutoResearcherConfig` class reads all; no component reaches `os.environ` directly |
| 3 | **Shared mutable state in async** | `CostMeter._costs` mutated from concurrent `asyncio.gather` tasks → race condition | Use `asyncio.Lock` around meter writes, or atomic `dict.update` patterns |
| 4 | **Tests hit the real API** | CI bill explodes; tests are slow and flaky on rate limits | `MockInferenceBackend` in conftest; real-API tests gated behind `--integration` flag |
| 5 | **Single `ComponentRegistry` per process** | Fine in prod, breaks in parallel pytest runs that share state | Make Registry stateless or scope it per-test via fixtures |
| 6 | **Swarm port collision** | Two test runs start A2A agents on `:9001` → `OSError: [Errno 98] Address already in use` | Randomize ports in test config; use `port=0` (OS assigns) in non-prod |
| 7 | **OTel `InMemorySpanExporter` OOM** | 10K requests → 10K spans in RAM → OOM | Switch to `BatchSpanProcessor` + OTLP export in prod; flush + clear in tests |
| 8 | **Version drift between A2A agents** | Searcher schema changes, orchestrator still sends old format | Version the `AgentCard.version` field; reject tasks with version mismatch |
| 9 | **pyproject.toml missing optional dep guard** | `pip install auto-researcher` succeeds, but `import auto_researcher.multimodal` crashes at runtime | Make top-level `__init__.py` never import sub-packages; lazy-load everything |
| 10 | **CI passes, Colab fails** | CI uses Python 3.11, Colab uses 3.10; walrus operator / match statement breaks | Pin `python_requires = ">=3.10"`; test matrix includes 3.10 |

## Homework

1. **Wire the MockInferenceBackend** — run `pytest tests/test_pipeline.py -v` in the `/content/auto_researcher_v2/` directory. All tests should pass with zero API calls.

2. **Add a second test** — write `test_cost_under_budget` that creates an `AutoResearcherConfig` with `max_cost_usd=0.00001` and verifies the pipeline either respects the cap or raises a `BudgetExceeded` error gracefully.

3. **Extend the config** — add a `SearchConfig(BaseModel)` with `n_subqueries: int = 3` and `search_model: str = HAIKU`, and use it in `_plan()` and `_search()`.

4. **GitHub repo** — create `https://github.com/<your-username>/auto-researcher`, push the scaffold, and add the `ANTHROPIC_API_KEY` secret. The CI workflow should run on your first push.

5. **README first draft** — write a README.md with: (a) one-line description, (b) 5-line quickstart, (c) architecture diagram (copy the ASCII from this notebook), (d) contributing guide stub. This README will be the first thing GitHub visitors see — make it impressive.

---

## Phase 5 Roadmap

| Lesson | Topic | Builds on |
|---|---|---|
| **L51 ✅** | Architecture & Scaffold | All Phase 4 tracks |
| **L52** | Advanced Retrieval | L20 (Vector DBs), L7 (RAG), L23 (capstone) |
| **L53** | Production Deployment | L47 (GPU autoscaling), L48 (OTel), L50 (capstone stack) |
| **L54** | Developer Experience | L15 (OSS publishing), L17 (evals), L24 (reliability gate) |
| **L55** | Capstone — Ship It | All of Phase 5 |

**The rule for Phase 5:** each lesson leaves the repo in a better, more publishable state than it found it. By L55 you'll have a real, useful, well-tested open-source AI engineering library — the kind of project that gets noticed.

---

*Next: L52 — Advanced Retrieval: semantic caching + hybrid search at scale.*